# Lesson 3.2: Stop conditions
*Build an AI Agent from Scratch with LangGraph · Chapter 3*

**Goal:** make the loop stop for the right reason: a good finish, plus hard stops for steps and tokens.

**What you will be able to do**
- name the four ways a run ends: finish, step limit, token budget, and guards that steer toward finish
- write the two guards: block an identical repeated write, nudge finish after a green health check
- replay a recorded run fairly, and measure before and after on the same samples
- pick a token budget that does not cut off good runs

**How this notebook runs.** Every model reply comes from Nightwatch's record/replay cache
(`nightwatch/runs/cache`, `mode="replay"`), so nothing here calls the API and nothing costs a token.

In [1]:
# Setup: find the course repo's nightwatch/ folder, load .env, check the key.
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

for cand in ["../../nightwatch", "../../../nightwatch", "nightwatch"]:   # relative to this notebook
    if (Path(cand) / "agent_v0.py").exists():
        NW = Path(cand).resolve()
        break
else:
    raise SystemExit("Can't find nightwatch/. Open this notebook from the course repo (lessons/<lesson>/notebook/).")
sys.path.insert(0, str(NW))

load_dotenv(find_dotenv(usecwd=True))          # the .env at the repo root, if there is one
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("nightwatch/ found:", (NW / "agent_v0.py").exists(), "| Python", sys.version.split()[0])
if HAS_KEY:
    print("GROQ_API_KEY loaded.")
else:
    print("No GROQ_API_KEY. Get a free key as in lesson 0.2 (console.groq.com -> API Keys) and put")
    print("GROQ_API_KEY=... in the .env file at the repo root.")
print("This notebook has no cell that calls the API: every run is a replay from nightwatch/runs/cache.")

nightwatch/ found: True | Python 3.13.16
GROQ_API_KEY loaded.
This notebook has no cell that calls the API: every run is a replay from nightwatch/runs/cache.


## 1. Recall: when did the 3.1 loop stop?

Only when the model answered with plain text and no tool call (step 8 in the slides). Nightwatch v0 adds a
`finish` tool and caps the loop at 14 steps; the guards are off by default.

In [2]:
import inspect, json
import agent_v0
from brightcart.incidents import generate

print(inspect.signature(agent_v0.run))
I = {x.id: x for x in generate()}

(inc, model='openai/gpt-oss-20b', mode='record', sample=0, max_steps=14, verbose=False, guards=False, token_budget=None) -> 'dict'


## 2. Key words

| Word | Meaning |
|---|---|
| stop condition | any rule that ends the loop |
| finish tool | the tool the model calls to say "done", with service, cause and summary |
| step limit | the most steps one run may take (Nightwatch: 14) |
| token budget | the most tokens one run may spend |
| guard | a check in your own code, between the model and the tools |
| idempotent write | same result run once or twice (set a value to 20); a restart is not |

## 3. The idea, then a hand trace

**Four ways a run ends:** (1) the model calls `finish`: the good stop; (2) the step limit: the loop ends anyway, and the
run counts as a failure; (3) the token budget: past it, we stop before the next call; (4) guards stop nothing
by themselves, they **steer**:
- guard one watches the 7 write tools: an identical repeated write (same tool, same arguments) is **not run**; the
  result says *ALREADY DONE ... run check_health*;
- guard two watches `check_health`: when it is green, one user message is added: *Health is green. **If** you have
  found the cause, call finish now.* The model still decides.

Hand trace: incident `pool_config-3`, sample 1 (checkout's DB pool size was cut from 20 to 2).

| step | before (no guards) | after (guards on) |
|---|---|---|
| 1–6 | status, logs, a malformed call, deploys, status, config_changes | the same replies (same requests) |
| 7 | `set_config(checkout, db_pool_size=20)` | the same |
| 8 | `check_health` → healthy True | the same, **then the nudge is appended** |
| 9 | ... keeps working | malformed call again |
| 10 | ... | `finish(bad_config, checkout)` → pass |
| end | 14 steps, no finish → fail | 10 steps, finish → pass |

**Is it fair?** Each reply is cached under a key: the exact request body plus the sample number. Same messages and
same sample give the same key, so the same reply. When a guard adds a message the body changes, and the model
answers fresh. So the two runs can only differ from the point where a guard spoke. See it:

In [3]:
import hashlib

def cache_key(body, sample):
    return hashlib.sha256((json.dumps(body, sort_keys=True) + f"#{sample}").encode()).hexdigest()[:24]

body = {"model": "openai/gpt-oss-20b", "messages": [{"role": "user", "content": "PAGER: x"}]}
nudged = {**body, "messages": body["messages"] + [{"role": "user", "content": "Health is green. ..."}]}
print("same body, same sample:", cache_key(body, 1) == cache_key(dict(body), 1))
print("same body, other sample:", cache_key(body, 1) == cache_key(body, 2))
print("one extra message      :", cache_key(body, 1) == cache_key(nudged, 1))

same body, same sample: True
same body, other sample: False
one extra message      : False


## 4. Write it yourself

The stop code of `agent_v0.run`, rebuilt as our own `my_run`. The request body, tools and system prompt are
agent_v0's, so our requests hit the same cache entries. Read the four marked places: step limit, budget, finish, guards.

In [4]:
from brightcart.incidents import fresh
from brightcart.sim import World


def write_key(name, args):
    return (name, json.dumps(args, sort_keys=True))     # same change -> same key, whatever the order


def my_run(inc, sample=0, max_steps=14, guards=False, token_budget=None, mode="replay"):
    w = World(fresh(inc))
    msgs = [{"role": "system", "content": agent_v0.SYSTEM}, {"role": "user", "content": f"PAGER: {inc.alert}"}]
    done_writes, trace, used, final, stop = {}, [], 0, None, None
    for step in range(max_steps):                                     # (1) the step limit
        if token_budget and used > token_budget:                      # (2) the budget, before every call
            stop = "token budget"
            break
        out = agent_v0.chat({"model": "openai/gpt-oss-20b", "messages": msgs, "tools": agent_v0.TOOLS,
                             "temperature": 1.0, "reasoning_effort": "medium", "max_completion_tokens": 2048},
                            mode, sample)
        if "error" in out:
            msgs.append({"role": "user", "content": "Your last tool call was malformed. Call one tool with valid JSON."})
            trace.append({"step": step, "error": out["error"]})
            continue
        u = out.get("usage", {})
        used += u.get("prompt_tokens", 0) + u.get("completion_tokens", 0)
        m = out["choices"][0]["message"]
        calls = m.get("tool_calls") or []
        msgs.append({"role": "assistant", "content": m.get("content") or "", **({"tool_calls": calls} if calls else {})})
        if not calls:
            msgs.append({"role": "user", "content": "Use the tools. End with finish()."})
            trace.append({"step": step, "text": (m.get("content") or "")[:200]})
            continue
        for c in calls:
            name = c["function"]["name"]
            try:
                args = json.loads(c["function"]["arguments"] or "{}")
            except json.JSONDecodeError:
                args = {}
            if name == "finish":                                      # (3) the good stop
                final, result = args, "ok"
            elif guards and name in agent_v0.WRITE_TOOLS and write_key(name, args) in done_writes:   # (4a) repeat guard
                result = (f"ALREADY DONE at step {done_writes[write_key(name, args)]}: "
                          "this exact change was applied. Do not repeat it; run check_health.")
            else:
                try:
                    result = agent_v0.run_tool(w, name, args)
                except Exception as e:
                    result = f"ERROR: {e}"
                if guards and name in agent_v0.WRITE_TOOLS:
                    done_writes[write_key(name, args)] = step
            trace.append({"step": step, "tool": name, "args": args, "result": str(result)[:300]})
            msgs.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(result, default=str)[:4000]})
        if final is not None:
            break
        if guards and any(t.get("tool") == "check_health" and "'healthy': True" in t.get("result", "")
                          for t in trace if t.get("step") == step):            # (4b) the green nudge
            msgs.append({"role": "user", "content": "Health is green. If you have found the cause, call finish now."})
    return {"stop_reason": stop or ("finish" if final is not None else "step limit"), "steps": len(trace),
            "final": final, "writes": w.actions, "healthy": w.health()["healthy"], "tokens": used, "trace": trace}

In [5]:
mine = my_run(I["pool_config-3"], sample=1, guards=True, token_budget=60000)
for t in mine["trace"]:
    what = f"{t['tool']}({json.dumps(t['args'])[:44]}) -> {t['result'][:50]}" if "tool" in t else "(malformed tool call)"
    print(f"{t['step'] + 1:2}  {what}")
print("stop:", mine["stop_reason"], "| steps:", mine["steps"], "| tokens:", mine["tokens"], "| final:", mine["final"]["cause"],
      mine["final"]["root_cause_service"])

 1  service_status({"service": "checkout"}) -> {'checkout': {'version': 'v2.7.3', 'replicas': 3, 
 2  read_logs({"n": 20, "service": "checkout"}) -> ['02:07 INFO checkout handled request id=58103 in 
 3  (malformed tool call)
 4  recent_deploys({}) -> [{'t': '02:01', 'service': 'inventory', 'version':
 5  service_status({"service": "inventory"}) -> {'inventory': {'version': 'v3.3.6', 'replicas': 2,
 6  config_changes({}) -> [{'t': '02:20', 'service': 'checkout', 'key': 'db_
 7  set_config({"key": "db_pool_size", "service": "checkout) -> set_config(checkout) done.
 8  check_health({}) -> {'healthy': True, 'unhealthy_services': [], 'queue
 9  (malformed tool call)
10  finish({"cause": "bad_config", "root_cause_service") -> ok
stop: finish | steps: 10 | tokens: 9499 | final: bad_config checkout


## 5. Library version + check

The "library" is `agent_v0.run(..., guards=True, token_budget=...)`. Same incident, same sample, same budget:
our loop must give the same stop reason, steps, writes and tokens.

In [6]:
for budget in (60000, 5000):
    lib = agent_v0.run(I["pool_config-3"], mode="replay", sample=1, guards=True, token_budget=budget)
    me = my_run(I["pool_config-3"], sample=1, guards=True, token_budget=budget)
    same = (lib["stop_reason"], lib["steps"], lib["writes"], sum(lib["usage"].values())) == \
           (me["stop_reason"], me["steps"], me["writes"], me["tokens"])
    print(f"budget {budget:5}: agent_v0 {lib['stop_reason']:12} steps {lib['steps']:2} tokens {sum(lib['usage'].values())}"
          f" | my_run {me['stop_reason']:12} steps {me['steps']:2} | same: {same}")

budget 60000: agent_v0 finish       steps 10 tokens 9499 | my_run finish       steps 10 | same: True
budget  5000: agent_v0 token budget steps  6 tokens 5285 | my_run token budget steps  6 | same: True


Measure it: the 10 incidents with the most no-finish runs, samples 0–2, 30 runs per side. "Before" is the saved
baseline (no guards); "after" replays the guarded runs (about a minute, still no API call).

In [7]:
from collections import Counter
rows = [json.loads(line) for line in open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8")]


def why(r):
    if r["pass"]:
        return "pass"
    if not r["final"]:
        return "no finish"
    if not r["healthy"]:
        return "unhealthy"
    return "wrong service" if not r["right_service"] else "extra writes"


print("baseline, 180 runs:", dict(Counter(why(r) for r in rows)))
worst = ["pool_config-3", "stuck_consumer-1", "stuck_consumer-5", "traffic_spike-1", "memory_leak-0",
         "memory_leak-1", "memory_leak-2", "memory_leak-4", "memory_leak-5", "stuck_consumer-0"]
before = [r for r in rows if r["incident"] in worst]
after = []
for r in before:
    a = agent_v0.run(I[r["incident"]], mode="replay", sample=r["sample"], guards=True, token_budget=60000)
    after.append(a | agent_v0.grade(a))
tok = lambda rs: sum(sum(r["usage"].values()) for r in rs) / len(rs)
print(f"pass: before {sum(r['pass'] for r in before)}/30 -> after {sum(r['pass'] for r in after)}/30")
print("failure kinds after:", dict(Counter(why(r) for r in after)))
print(f"mean steps {sum(r['steps'] for r in before) / 30:.1f} -> {sum(r['steps'] for r in after) / 30:.1f} | "
      f"mean tokens {tok(before):,.0f} -> {tok(after):,.0f}")
print("blocked repeats:", sum(str(t.get('result', '')).startswith('ALREADY DONE') for a in after for t in a['trace']),
      "| runs stopped by the 60,000 budget:", sum(a['stop_reason'] == 'token budget' for a in after))

baseline, 180 runs: {'pass': 124, 'no finish': 46, 'extra writes': 9, 'wrong service': 1}


pass: before 5/30 -> after 15/30
failure kinds after: {'pass': 15, 'extra writes': 1, 'no finish': 14}
mean steps 13.5 -> 12.7 | mean tokens 14,475 -> 14,286
blocked repeats: 4 | runs stopped by the 60,000 budget: 0


5 of 30 (17%) became 15 of 30 (50%). What the guards did not fix: `stuck_consumer-1` and `traffic_spike-1` hit the
step limit in all six runs; in five of them the agent never saw a green check, so the nudge never fired.

In [8]:
for iid in ["stuck_consumer-1", "traffic_spike-1"]:
    for s in range(3):
        a = agent_v0.run(I[iid], mode="replay", sample=s, guards=True, token_budget=60000)
        g = sum(1 for t in a["trace"] if t.get("tool") == "check_health" and "'healthy': True" in t.get("result", ""))
        print(f"{iid:17} s{s}  stop={a['stop_reason']:10}  green health checks {g}")

stuck_consumer-1  s0  stop=step limit  green health checks 0
stuck_consumer-1  s1  stop=step limit  green health checks 0
stuck_consumer-1  s2  stop=step limit  green health checks 0
traffic_spike-1   s0  stop=step limit  green health checks 1
traffic_spike-1   s1  stop=step limit  green health checks 0
traffic_spike-1   s2  stop=step limit  green health checks 0


Reaching a checked fix is a diagnosis problem (later chapters), not a stopping problem.

## 6. Where it shows up in agents

Every production agent needs a hard cap and a good way to finish. In chapter 4, LangGraph's **recursion limit** is the
same step cap, enforced by the framework. A two-node graph that never stops, capped at 5 steps (no model involved):

In [9]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.errors import GraphRecursionError


class S(TypedDict):
    n: int


g = StateGraph(S)
g.add_node("think", lambda s: {"n": s["n"] + 1})
g.add_node("act", lambda s: {"n": s["n"] + 1})
g.add_edge(START, "think")
g.add_edge("think", "act")
g.add_edge("act", "think")              # no way out: the graph never ends by itself
try:
    g.compile().invoke({"n": 0}, config={"recursion_limit": 5})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e)[:90])

GraphRecursionError: Recursion limit of 5 reached without hitting a stop condition. You can increase the limit 


## 7. Traps

**Trap 1: only a step cap.** It does stop the loop, but every stop it makes is a failure.

In [10]:
capped = [r for r in rows if r["steps"] == 14 and not r["final"]]
print(f"{len(capped)} runs stopped by the cap; passed: {sum(r['pass'] for r in capped)}; "
      f"ended healthy anyway: {sum(r['healthy'] for r in capped)}")

46 runs stopped by the cap; passed: 0; ended healthy anyway: 31


**Trap 2: nudging before anything is verified.** A nudge on every step lets the model declare victory on a broken
shop. Nudge only after a green `check_health` in *this* step:

In [11]:
red_step = [{"step": 0, "tool": "check_health", "result": "{'healthy': False, 'unhealthy_services': ['checkout']}"}]
green_step = [{"step": 8, "tool": "check_health", "result": "{'healthy': True, 'unhealthy_services': []}"}]
nudge_always = lambda entries: True                      # the trap
nudge_green = lambda entries: any(t.get("tool") == "check_health" and "'healthy': True" in t["result"] for t in entries)
print("broken shop -> always:", nudge_always(red_step), "| green only:", nudge_green(red_step))
print("fixed shop  -> always:", nudge_always(green_step), "| green only:", nudge_green(green_step))

broken shop -> always: True | green only: False
fixed shop  -> always: True | green only: True


**Trap 3: blocking repeated reads.** Reading the logs twice is fine; only identical *writes* are blocked.

In [12]:
for name in ["read_logs", "restart_service"]:
    print(f"{name:16} is a write tool: {name in agent_v0.WRITE_TOOLS} -> repeat "
          f"{'blocked' if name in agent_v0.WRITE_TOOLS else 'allowed'}")

read_logs        is a write tool: False -> repeat allowed
restart_service  is a write tool: True -> repeat blocked


**Trap 4: a budget that is too low** cuts good runs off. The same pool_config-3 run with smaller budgets:

In [13]:
for b in (5000, 7000, 9000):
    x = agent_v0.run(I["pool_config-3"], mode="replay", sample=1, guards=True, token_budget=b)
    print(f"budget {b}: stop={x['stop_reason']:12} steps {x['steps']:2} writes {x['writes']} pass {agent_v0.grade(x)['pass']}")

budget 5000: stop=token budget steps  6 writes [] pass False
budget 7000: stop=token budget steps  8 writes [('set_config', 'checkout', 'db_pool_size=20')] pass False
budget 9000: stop=finish       steps 10 writes [('set_config', 'checkout', 'db_pool_size=20')] pass True


With 7,000 the fix is made, but the run is cut before `finish`: a healthy shop and still a fail.

## 8. Exercises (no API: every check replays from the cache)

In [14]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

**Exercise 1 (the video's exercise).** Replay `pool_config-3`, sample 1, with guards on and `token_budget=5000`
(use `agent_v0.run` in replay mode). Store the run in `r5`. Predict the stop reason and the steps first.

In [15]:
r5 = None
# YOUR CODE HERE

In [16]:
def test1():
    assert r5 is not None, "store the run in r5"
    assert r5["stop_reason"] == "token budget", f"expected the budget stop, got {r5['stop_reason']}"
    assert r5["steps"] == 6, f"expected 6 steps, got {r5['steps']}"
    assert sum(r5["usage"].values()) == 5285, "expected 5285 tokens used"
    assert r5["writes"] == [], "it stopped while still reading: no write yet"
check(1, test1)

✗ Exercise 1 failed: store the run in r5


**Exercise 2.** Write `same_write(a, b)`: two write calls `(name, args)` are the same change when the name is equal and
the arguments are equal **whatever their key order**. Use sorted JSON.

In [17]:
def same_write(a, b):
    # YOUR CODE HERE
    pass

In [18]:
def test2():
    x = ("set_config", {"service": "checkout", "key": "db_pool_size", "value": "20"})
    y = ("set_config", {"value": "20", "key": "db_pool_size", "service": "checkout"})
    z = ("set_config", {"service": "checkout", "key": "db_pool_size", "value": "2"})
    assert same_write(x, y) is True, "same change, keys in another order"
    assert same_write(x, z) is False, "a different value is a different change"
    assert same_write(("restart_service", {"service": "web"}), ("rollback_deploy", {"service": "web"})) is False
check(2, test2)

✗ Exercise 2 failed: same change, keys in another order


**Exercise 3.** Write `green_steps(trace)`: the list of step numbers (from each entry's `"step"`) where a
`check_health` result contains `'healthy': True`. Those are the steps after which guard two nudges.

In [19]:
def green_steps(trace):
    # YOUR CODE HERE
    pass

In [20]:
def test3():
    run = agent_v0.run(I["pool_config-3"], mode="replay", sample=1, guards=True, token_budget=60000)
    assert green_steps(run["trace"]) == [7], f"the green check is at step 7 (counting from 0), got {green_steps(run['trace'])}"
    assert green_steps([{"step": 0, "error": "tool_use_failed"}]) == [], "malformed entries have no tool"
    assert green_steps(red_step + green_step) == [8]
check(3, test3)

✗ Exercise 3 failed: the green check is at step 7 (counting from 0), got None


**Exercise 4.** Write `smallest_budget(budgets)`: replay `pool_config-3`, sample 1, guards on, for each budget in the
list (smallest first) and return the first one whose run **passes** `agent_v0.grade`. Return None if none does.

In [21]:
def smallest_budget(budgets):
    # YOUR CODE HERE
    pass

In [22]:
def test4():
    got = smallest_budget([5000, 6000, 7000, 8000, 9000, 10000])
    assert got == 9000, f"expected 9000 (7000 and 8000 fix the shop but stop before finish), got {got}"
    assert smallest_budget([5000, 6000]) is None, "neither 5000 nor 6000 lets the run finish"
check(4, test4)

✗ Exercise 4 failed: expected 9000 (7000 and 8000 fix the shop but stop before finish), got None


## Solutions (try first!)

In [23]:
r5 = agent_v0.run(I["pool_config-3"], mode="replay", sample=1, guards=True, token_budget=5000)
print(r5["stop_reason"], r5["steps"], sum(r5["usage"].values()))

token budget 6 5285


In [24]:
def same_write(a, b):
    return a[0] == b[0] and json.dumps(a[1], sort_keys=True) == json.dumps(b[1], sort_keys=True)

In [25]:
def green_steps(trace):
    return [t["step"] for t in trace
            if t.get("tool") == "check_health" and "'healthy': True" in t.get("result", "")]

In [26]:
def smallest_budget(budgets):
    for b in sorted(budgets):
        x = agent_v0.run(I["pool_config-3"], mode="replay", sample=1, guards=True, token_budget=b)
        if agent_v0.grade(x)["pass"]:
            return b
    return None